<a href="https://colab.research.google.com/github/Uchihatt/My-Projects/blob/main/Business_Model_Test_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor


In [ ]:
orders = pd.read_csv("ecommerce_orders.csv")
products = pd.read_csv("ecommerce_products.csv")
customers = pd.read_csv("ecommerce_customers.csv")

orders.head()


,order_id,order_date,customer_id,product_id,units,channel,region,device,is_new_customer,discount_pct,shipping_cost,returned,list_price,price_paid,revenue,net_revenue,cogs,gross_margin,net_margin
0,1,2024-02-02,11857,230,1,Direct,South,Mobile,0,0.0371,5.75,0,33.67,32.42,32.42,32.42,15.21,11.46,11.46
1,2,2024-10-09,12487,227,2,Paid Search,North,Desktop,0,0.0856,5.59,0,51.56,47.15,94.29,94.29,70.42,18.29,18.29
2,3,2024-08-26,11920,204,3,Social,South,Desktop,0,0.1155,6.71,0,49.07,43.40,130.21,130.21,98.19,25.31,25.31
3,4,2024-06-09,11782,207,2,Paid Search,West,Desktop,0,0.0968,8.48,0,38.54,34.81,69.62,69.62,54.68,6.46,6.46
4,5,2024-06-07,11402,235,1,Paid Search,South,Desktop,0,0.0728,4.69,0,35.04,32.49,32.49,32.49,20.03,7.77,7.77


In [ ]:
# @title Missing Values Check
orders["order_date"] = pd.to_datetime(orders["order_date"])
customers["signup_date"] = pd.to_datetime(customers["signup_date"])
print("Orders nulls:\n", orders.isna().sum().sort_values(ascending=False).head(10))
print("\nProducts nulls:\n", products.isna().sum().sort_values(ascending=False).head(10))
print("\nCustomers nulls:\n", customers.isna().sum().sort_values(ascending=False).head(10))


Orders nulls:
 order_id           0
order_date         0
customer_id        0
product_id         0
units              0
channel            0
region             0
device             0
is_new_customer    0
discount_pct       0
dtype: int64

Products nulls:
 product_id    0
category      0
brand         0
list_price    0
unit_cost     0
dtype: int64

Customers nulls:
 customer_id    0
signup_date    0
age            0
segment        0
city           0
dtype: int64


In [ ]:
# @title Joins
df = orders.merge(products, on="product_id", how="left") \
           .merge(customers, on="customer_id", how="left")

df.shape, df.head()



((6000, 27),
    order_id order_date  customer_id  product_id  units      channel region  \
 0         1 2024-02-02        11857         230      1       Direct  South   
 1         2 2024-10-09        12487         227      2  Paid Search  North   
 2         3 2024-08-26        11920         204      3       Social  South   
 3         4 2024-06-09        11782         207      2  Paid Search   West   
 4         5 2024-06-07        11402         235      1  Paid Search  South   
 
     device  is_new_customer  discount_pct  ...  gross_margin  net_margin  \
 0   Mobile                0        0.0371  ...         11.46       11.46   
 1  Desktop                0        0.0856  ...         18.29       18.29   
 2  Desktop                0        0.1155  ...         25.31       25.31   
 3  Desktop                0        0.0968  ...          6.46        6.46   
 4  Desktop                0        0.0728  ...          7.77        7.77   
 
       category  brand  list_price_y  unit_cost

In [ ]:
print("Missing product rows:", df["category"].isna().sum())
print("Missing customer rows:", df["segment"].isna().sum())

Missing product rows: 0
Missing customer rows: 0


In [ ]:
df["order_month"] = df["order_date"].dt.month
df["order_dayofweek"] = df["order_date"].dt.dayofweek
df["customer_tenure_days"] = (df["order_date"] - df["signup_date"]).dt.days
df["customer_tenure_days"] = df["customer_tenure_days"].clip(lower=0)
df["shipping_per_unit"] = df["shipping_cost"] / df["units"]
df["shipping_per_unit"] = df["shipping_per_unit"].replace([np.inf, -np.inf], np.nan).fillna(df["shipping_per_unit"].median())
target = "net_margin"


In [ ]:
# @title Outlier Handling
low, high = df[target].quantile([0.01, 0.99])
df[target] = df[target].clip(low, high)


In [ ]:
# @title Feature Handling
features = [
    "units",
    "discount_pct",
    "shipping_cost",
    "shipping_per_unit",
    "returned",
    "channel",
    "region",
    "device",
    "is_new_customer",
    "category",
    "brand",
    "segment",
    "city",
    "order_month",
    "order_dayofweek",
    "customer_tenure_days",
    "list_price",
    "unit_cost"
]

X = df[features]
y = df[target]


In [ ]:
# If merge created duplicates, normalize them:
if "list_price" not in df.columns:
    if "list_price_x" in df.columns and "list_price_y" in df.columns:
        df["list_price"] = df["list_price_y"]  # choose products version
    elif "list_price_x" in df.columns:
        df["list_price"] = df["list_price_x"]
    elif "list_price_y" in df.columns:
        df["list_price"] = df["list_price_y"]

if "unit_cost" not in df.columns:
    if "unit_cost_x" in df.columns and "unit_cost_y" in df.columns:
        df["unit_cost"] = df["unit_cost_y"]  # choose products version
    elif "unit_cost_x" in df.columns:
        df["unit_cost"] = df["unit_cost_x"]
    elif "unit_cost_y" in df.columns:
        df["unit_cost"] = df["unit_cost_y"]


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [ ]:
cat_cols = X.select_dtypes(include=["object"]).columns.tolist()
num_cols = [c for c in X.columns if c not in cat_cols]


In [ ]:
preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)
    ]
)


In [ ]:
# @title Train Models
def evaluate(model, name):
    pipe = Pipeline(steps=[("prep", preprocess), ("model", model)])
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)

    mae = mean_absolute_error(y_test, preds)
    r2 = r2_score(y_test, preds)

    print(f"{name}: MAE={mae:.2f}, R2={r2:.3f}")
    return pipe


In [ ]:
pipes = {}
pipes["Linear"] = evaluate(LinearRegression(), "Linear")
pipes["Ridge"] = evaluate(Ridge(alpha=1.0), "Ridge")
pipes["Lasso"] = evaluate(Lasso(alpha=0.01), "Lasso")
pipes["ElasticNet"] = evaluate(ElasticNet(alpha=0.01, l1_ratio=0.5), "ElasticNet")
pipes["RF"] = evaluate(RandomForestRegressor(n_estimators=300, random_state=42), "RandomForest")


Linear: MAE=3.57, R2=0.902
Ridge: MAE=3.56, R2=0.902
Lasso: MAE=3.55, R2=0.902
ElasticNet: MAE=3.50, R2=0.903
RandomForest: MAE=1.50, R2=0.976


In [ ]:
# @title Get Top Drivers
best = pipes["Ridge"]

ohe = best.named_steps["prep"].named_transformers_["cat"]
cat_feature_names = ohe.get_feature_names_out(cat_cols)

feature_names = np.concatenate([num_cols, cat_feature_names])
coefs = best.named_steps["model"].coef_

coef_df = pd.DataFrame({"feature": feature_names, "coef": coefs})
coef_df["abs_coef"] = coef_df["coef"].abs()
coef_df.sort_values("abs_coef", ascending=False).head(20)



,feature,coef,abs_coef
9,list_price,21.434286,21.434286
10,unit_cost,-16.818116,16.818116
0,units,11.585104,11.585104
1,discount_pct,-3.461792,3.461792
4,returned,-1.850487,1.850487
2,shipping_cost,-0.720951,0.720951
3,shipping_per_unit,-0.501162,0.501162
28,category_Home,-0.379956,0.379956
12,channel_Direct,0.333524,0.333524
23,device_Tablet,-0.300349,0.300349


In [ ]:
# @title Business what-if: “Reduce discount by 2% on Paid Search”
best_model = pipes["Ridge"]  # or whichever you choose

X_test_base = X_test.copy()
X_test_scenario = X_test.copy()

mask = (X_test_scenario["channel"] == "Paid Search")
X_test_scenario.loc[mask, "discount_pct"] = (X_test_scenario.loc[mask, "discount_pct"] - 0.02).clip(lower=0)
pred_base = best_model.predict(X_test_base)
pred_scn = best_model.predict(X_test_scenario)

uplift_per_order = pred_scn - pred_base
avg_uplift_paid_search = uplift_per_order[mask.values].mean()
total_uplift_paid_search = uplift_per_order[mask.values].sum()

avg_uplift_overall = uplift_per_order.mean()

avg_uplift_paid_search, total_uplift_paid_search, avg_uplift_overall
print(f"Scenario: Reduce discount by 2% (absolute) for Paid Search orders")
print(f"Avg net margin uplift per Paid Search order: {avg_uplift_paid_search:.2f}")
print(f"Total net margin uplift across Paid Search test orders: {total_uplift_paid_search:.2f}")
print(f"Avg uplift across all test orders: {avg_uplift_overall:.2f}")



Scenario: Reduce discount by 2% (absolute) for Paid Search orders
Avg net margin uplift per Paid Search order: 1.59
Total net margin uplift across Paid Search test orders: 424.08
Avg uplift across all test orders: 0.35


In [ ]:
#SQL
!pip -q install duckdb
import duckdb
con = duckdb.connect()

# Read CSVs as SQL tables (views)
con.execute("CREATE VIEW orders AS SELECT * FROM read_csv_auto('ecommerce_orders.csv');")
con.execute("CREATE VIEW products AS SELECT * FROM read_csv_auto('ecommerce_products.csv');")
con.execute("CREATE VIEW customers AS SELECT * FROM read_csv_auto('ecommerce_customers.csv');")

# Test query
con.execute("SELECT COUNT(*) AS n_orders FROM orders").df()

feature_df = con.execute("""
WITH base AS (
  SELECT
    o.order_id,
    CAST(o.order_date AS DATE) AS order_date,
    o.customer_id,
    o.product_id,
    o.net_margin,
    o.discount_pct,
    o.shipping_cost,
    o.returned,
    o.channel,
    o.region,
    o.device,
    o.is_new_customer,
    p.category,
    p.brand,
    c.segment,
    c.city,
    c.age,
    EXTRACT('month' FROM CAST(o.order_date AS DATE)) AS order_month,
    EXTRACT('dow' FROM CAST(o.order_date AS DATE)) AS order_dayofweek,
    DATE_DIFF('day', CAST(c.signup_date AS DATE), CAST(o.order_date AS DATE)) AS customer_tenure_days,
    (o.shipping_cost / NULLIF(o.units, 0)) AS shipping_per_unit,
    o.units,
    o.list_price,
    p.unit_cost
  FROM orders o
  LEFT JOIN products p ON o.product_id = p.product_id
  LEFT JOIN customers c ON o.customer_id = c.customer_id
)
SELECT * FROM base
""").df()

feature_df.head()


,order_id,order_date,customer_id,product_id,net_margin,discount_pct,shipping_cost,returned,channel,region,...,segment,city,age,order_month,order_dayofweek,customer_tenure_days,shipping_per_unit,units,list_price,unit_cost
0,1,2024-02-02,11857,230,11.46,0.0371,5.75,0,Direct,South,...,Premium,Karachi,26,2,5,-59,5.750000,1,33.67,15.21
1,2,2024-10-09,12487,227,18.29,0.0856,5.59,0,Paid Search,North,...,Mainstream,Faisalabad,62,10,3,-63,2.795000,2,51.56,35.21
2,3,2024-08-26,11920,204,25.31,0.1155,6.71,0,Social,South,...,Mainstream,Karachi,57,8,1,167,2.236667,3,49.07,32.73
3,4,2024-06-09,11782,207,6.46,0.0968,8.48,0,Paid Search,West,...,Value,Karachi,23,6,0,444,4.240000,2,38.54,27.34
4,5,2024-06-07,11402,235,7.77,0.0728,4.69,0,Paid Search,South,...,Value,Lahore,24,6,5,192,4.690000,1,35.04,20.03
